In [14]:
import pandas as pd
import requests
from tqdm import tqdm
import os

df = pd.read_csv(r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\raw\beijing_drink_shops_amap.csv")


test_df = df.sample(10).copy()

In [15]:
def get_nearby_poi_count(api_key, lon, lat, keyword, radius=500):
    
    url = "https://restapi.amap.com/v3/place/around"
    params = {
        'key': api_key,
        'location': f"{lon},{lat}",
        'keywords': keyword,
        'radius': radius
    }

    response = requests.get(url, params=params)
    result = response.json()

    if result.get('status') == '1':
        return int(result.get('count',0))
    else:
        return 0


In [ ]:
tqdm.pandas()

MY_KEY = '0e971d578b21c1bdb0e07daf1650576b'

df['office_count'] = df.progress_apply(lambda row:  get_nearby_poi_count(MY_KEY, row['lon'], row['lat'], '办公楼|写字楼', radius=500), axis=1)
df['park_count'] = df.progress_apply(lambda row:  get_nearby_poi_count(MY_KEY, row['lon'], row['lat'], '公园|景点', radius=500), axis=1)


def define_location_type(row):
    # 如果没啥写字楼，反而公园多，那是典型的旅游店（最不抗冻）
    if row['park_count'] > 5 and row['office_count'] < row['park_count']:
        return "脆弱：极度依赖天气"
    # 如果写字楼极其密集（比如大几十个），妥妥的室内容易活
    elif row['office_count'] > 20:
        return "抗寒：重度商务刚需"
    # 啥也不突出的就是社区型
    else:
        return "中庸：混合社区"
    
df['location_type'] = df.apply( define_location_type, axis=1)

100%|██████████| 557/557 [02:38<00:00,  3.52it/s]


In [ ]:

display(df.head())

output_path = r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\processed"
if not os.path.exists(output_path):
    os.makedirs(output_path)

df.to_csv(os.path.join(output_path,"beijing_drink_shops_processed.csv"), index=False, encoding='utf-8-sig')